In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import warnings, os, gc
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 10
EPOCHS = 200
BATCH_SIZE = 2048
LR = 1e-3
HIDDEN = [512, 512, 256, 256, 128]
DROPOUT = 0.3
LABEL_SMOOTH = 0.05
TARGET = 'Irrigation_Need'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda


In [9]:
COMP = 'data/raw/'
ORIG = 'data/OG_data/'

train = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/train.csv')
test  = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/test.csv')
orig  = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/OG_data/irrigation_prediction.csv')

target_map = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)

# Add original data
orig_aligned = orig.copy()
orig_aligned['id'] = -1
train_full = pd.concat([train, orig_aligned], ignore_index=True)

CATS = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season',
        'Irrigation_Type','Water_Source','Mulching_Used','Region']
NUMS = ['Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
        'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
        'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm']

print(f'Train: {train.shape}, Test: {test.shape}, Orig: {orig.shape}')
print(f'Train+Orig: {train_full.shape}')

Train: (630000, 21), Test: (270000, 20), Orig: (10000, 20)
Train+Orig: (640000, 21)


In [10]:
def engineer(df_list):
    for df in df_list:
        # Key domain interactions (from logit analysis)
        eps = 1e-6
        df['water_avail']    = df['Soil_Moisture'] + df['Rainfall_mm'] / 300.0
        df['heat_stress']    = df['Temperature_C'] / (df['Soil_Moisture'] + eps)
        df['rain_cool']      = df['Rainfall_mm'] / (df['Temperature_C'] + eps)
        df['wind_evap']      = df['Wind_Speed_kmh'] * df['Temperature_C'] / 100.0
        df['soil_x_rain']    = df['Soil_Moisture'] * df['Rainfall_mm'] / 1000.0
        df['temp_x_dry']     = df['Temperature_C'] * (100 - df['Soil_Moisture']) / 100.0
        df['soil_lt_25']     = (df['Soil_Moisture'] < 25).astype(int)
        df['temp_gt_30']     = (df['Temperature_C'] > 30).astype(int)
        df['rain_lt_300']    = (df['Rainfall_mm'] < 300).astype(int)
        df['wind_gt_10']     = (df['Wind_Speed_kmh'] > 10).astype(int)
    return df_list

engineer([train_full, test])

NEW_NUMS = ['water_avail','heat_stress','rain_cool','wind_evap',
            'soil_x_rain','temp_x_dry','soil_lt_25','temp_gt_30',
            'rain_lt_300','wind_gt_10']
ALL_NUMS = NUMS + NEW_NUMS
print(f'Total numeric features: {len(ALL_NUMS)}')

Total numeric features: 21


In [11]:
les = {}
for c in CATS:
    le = LabelEncoder()
    combined = pd.concat([train_full[c], test[c]]).astype(str)
    le.fit(combined)
    train_full[c] = le.transform(train_full[c].astype(str))
    test[c]       = le.transform(test[c].astype(str))
    les[c] = le

# Embedding dims: min(50, (n_unique // 2) + 1)
emb_dims = {c: (int(train_full[c].nunique()), min(50, int(train_full[c].nunique()) // 2 + 1))
            for c in CATS}
print('Embedding dims:', emb_dims)

Embedding dims: {'Soil_Type': (4, 3), 'Crop_Type': (6, 4), 'Crop_Growth_Stage': (4, 3), 'Season': (3, 2), 'Irrigation_Type': (4, 3), 'Water_Source': (4, 3), 'Mulching_Used': (2, 2), 'Region': (5, 3)}


In [12]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return x + self.net(x)


class TabMLP(nn.Module):
    def __init__(self, n_num, emb_dims, hidden, dropout, n_classes=3):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(n_cats, emb_dim) for n_cats, emb_dim in emb_dims.values()
        ])
        emb_total = sum(e for _, e in emb_dims.values())
        in_dim = n_num + emb_total

        layers = [nn.Linear(in_dim, hidden[0]), nn.GELU(), nn.Dropout(dropout)]
        for i in range(1, len(hidden)):
            layers += [nn.Linear(hidden[i-1], hidden[i]), nn.GELU(), nn.Dropout(dropout)]
        self.input_proj = nn.Sequential(*layers)

        self.res_blocks = nn.ModuleList([ResBlock(hidden[-1], dropout) for _ in range(3)])
        self.head = nn.Sequential(nn.LayerNorm(hidden[-1]), nn.Linear(hidden[-1], n_classes))

    def forward(self, x_num, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_num] + embs, dim=1)
        x = self.input_proj(x)
        for blk in self.res_blocks:
            x = blk(x)
        return self.head(x)

In [13]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, n_classes, smoothing=0.05, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.n_classes = n_classes
        self.weight = weight

    def forward(self, pred, target):
        confidence = 1.0 - self.smoothing
        smooth_val  = self.smoothing / (self.n_classes - 1)
        one_hot = torch.zeros_like(pred).scatter_(1, target.unsqueeze(1), 1)
        smooth_one_hot = one_hot * confidence + (1 - one_hot) * smooth_val
        log_prob = torch.log_softmax(pred, dim=1)
        if self.weight is not None:
            w = self.weight[target]
            loss = -(smooth_one_hot * log_prob).sum(dim=1)
            return (loss * w).mean()
        return -(smooth_one_hot * log_prob).sum(dim=1).mean()

In [14]:
def get_tensors(df, scaler=None, fit_scaler=False):
    X_num = df[ALL_NUMS].values.astype(np.float32)
    X_cat = df[CATS].values.astype(np.int64)
    if fit_scaler:
        scaler = StandardScaler()
        X_num = scaler.fit_transform(X_num)
    else:
        X_num = scaler.transform(X_num)
    return torch.FloatTensor(X_num), torch.LongTensor(X_cat), scaler


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x_num, x_cat, y in loader:
        x_num, x_cat, y = x_num.to(DEVICE), x_cat.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x_num, x_cat)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs = []
    for x_num, x_cat in loader:
        x_num, x_cat = x_num.to(DEVICE), x_cat.to(DEVICE)
        out = torch.softmax(model(x_num, x_cat), dim=1)
        probs.append(out.cpu().numpy())
    return np.vstack(probs)

In [15]:
# Only train rows (not orig) for fold splitting
train_idx = train_full[train_full['id'] >= 0].index
orig_idx  = train_full[train_full['id'] < 0].index

X_train = train_full.loc[train_idx].reset_index(drop=True)
y_train = X_train[TARGET].values
X_orig  = train_full.loc[orig_idx].reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs  = np.zeros((len(X_train), 3))
test_probs = np.zeros((len(test), 3))

class_weights_arr = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y_train)
cw_tensor = torch.FloatTensor(class_weights_arr).to(DEVICE)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
    Xtr = pd.concat([X_train.iloc[tr_idx], X_orig], ignore_index=True)
    ytr = np.concatenate([y_train[tr_idx], X_orig[TARGET].values])
    Xval = X_train.iloc[val_idx]
    yval = y_train[val_idx]

    tr_num, tr_cat, scaler = get_tensors(Xtr, fit_scaler=True)
    val_num, val_cat, _    = get_tensors(Xval, scaler=scaler)
    te_num,  te_cat,  _    = get_tensors(test, scaler=scaler)

    tr_ds  = TensorDataset(tr_num, tr_cat, torch.LongTensor(ytr))
    val_ds = TensorDataset(val_num, val_cat)
    te_ds  = TensorDataset(te_num, te_cat)

    tr_loader  = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=4096,  shuffle=False, num_workers=2)
    te_loader  = DataLoader(te_ds, batch_size=4096,  shuffle=False, num_workers=2)

    model = TabMLP(len(ALL_NUMS), emb_dims, HIDDEN, DROPOUT).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    criterion = LabelSmoothingLoss(3, LABEL_SMOOTH, weight=cw_tensor)

    best_score, patience, best_state = 0, 20, None
    for epoch in range(EPOCHS):
        train_epoch(model, tr_loader, optimizer, criterion)
        scheduler.step()
        val_prob = predict_proba(model, val_loader)
        score = balanced_accuracy_score(yval, val_prob.argmax(1))
        if score > best_score:
            best_score = score
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience = 20
        else:
            patience -= 1
            if patience == 0:
                break

    model.load_state_dict(best_state)
    oof_probs[val_idx] = predict_proba(model, val_loader)
    test_probs += predict_proba(model, te_loader) / N_FOLDS
    print(f'  Fold {fold+1} best balanced accuracy: {best_score:.5f}')
    del model; gc.collect(); torch.cuda.empty_cache()

oof_score = balanced_accuracy_score(y_train, oof_probs.argmax(1))
print(f'\nOOF Balanced Accuracy: {oof_score:.5f}')


=== Fold 1/10 ===
  Fold 1 best balanced accuracy: 0.96438

=== Fold 2/10 ===
  Fold 2 best balanced accuracy: 0.96528

=== Fold 3/10 ===
  Fold 3 best balanced accuracy: 0.96344

=== Fold 4/10 ===
  Fold 4 best balanced accuracy: 0.96572

=== Fold 5/10 ===
  Fold 5 best balanced accuracy: 0.96541

=== Fold 6/10 ===
  Fold 6 best balanced accuracy: 0.96544

=== Fold 7/10 ===
  Fold 7 best balanced accuracy: 0.96464

=== Fold 8/10 ===
  Fold 8 best balanced accuracy: 0.96305

=== Fold 9/10 ===
  Fold 9 best balanced accuracy: 0.96469

=== Fold 10/10 ===
  Fold 10 best balanced accuracy: 0.96398

OOF Balanced Accuracy: 0.96460


In [17]:
# Hard labels submission
preds = test_probs.argmax(axis=1)
sub = pd.read_csv('/home/chotu/Downloads/Irrigation_Submission/data/raw/sample_submission.csv')
sub[TARGET] = [reverse_map[p] for p in preds]
sub.to_csv('submission_nn.csv', index=False)
print('submission_nn.csv saved')
print(sub[TARGET].value_counts())

# Soft probabilities (USE THESE FOR ENSEMBLING!)
proba_df = pd.DataFrame(test_probs, columns=['proba_Low','proba_Medium','proba_High'])
proba_df.insert(0, 'id', sub['id'].values)
proba_df.to_csv('soft_proba_nn.csv', index=False)
print('soft_proba_nn.csv saved — use this for soft voting ensemble!')

# OOF (for stacking)
oof_df = pd.DataFrame(oof_probs, columns=['oof_Low','oof_Medium','oof_High'])
oof_df['true'] = y_train
oof_df.to_csv('oof_nn.csv', index=False)
print('oof_nn.csv saved')

submission_nn.csv saved
Irrigation_Need
Low       159923
Medium    100217
High        9860
Name: count, dtype: int64
soft_proba_nn.csv saved — use this for soft voting ensemble!
oof_nn.csv saved
